# Analyse exploratoire — `eval_database.json`
## Dataset d'optimisation BWB Flying Wing (nEUROn v2)

**Contenu du dataset :**
- **X** : 10 000 échantillons × 30 variables de design (planform, centre-corps, twist, dièdre, profil Kulfan CST)
- **results** : coefficients aérodynamiques, performances, contraintes, faisabilité

### Plan d'analyse
1. Chargement & mise en forme
2. Statistiques descriptives des variables de design
3. Statistiques descriptives des réponses (outputs)
4. Analyse de faisabilité & contraintes
5. Distributions & outliers
6. Corrélations (inputs ↔ outputs)
7. Analyse en composantes principales (PCA)
8. Analyse des designs faisables vs infaisables
9. Visualisations avancées (parallel coordinates, pairplots)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "white"})

# ── Chargement ──
# Robuste : fonctionne depuis notebooks/ ou depuis la racine
_candidates = [Path("../data/eval_database.json"), Path("data/eval_database.json")]
data_path = next(p for p in _candidates if p.exists())
with open(data_path) as f:
    raw = json.load(f)

X_raw = np.array(raw["X"])          # (10000, 30)
results_list = raw["results"]       # list of 10000 dicts

print(f"X shape : {X_raw.shape}")
print(f"Nb results : {len(results_list)}")
print(f"Clés result : {list(results_list[0].keys())}")

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Noms des 30 variables de design (ordre design_variables.py) ──
VAR_NAMES = [
    # Planform (4)
    "half_span", "wing_root_chord", "taper_ratio", "le_sweep_deg",
    # Centre-corps (8)
    "body_chord_ratio", "body_halfwidth", "body_tc_root", "body_camber",
    "body_reflex", "body_twist", "body_sweep_delta", "body_le_droop",
    # Twist (5)
    "twist_1", "twist_2", "twist_3", "twist_4", "twist_tip",
    # Dièdre (5)
    "dihedral_0", "dihedral_1", "dihedral_2", "dihedral_3", "dihedral_tip",
    # Kulfan CST (8)
    "kulfan_root_u2", "kulfan_root_u6", "kulfan_root_u7",
    "kulfan_root_l2", "kulfan_root_l6", "kulfan_root_l7",
    "kulfan_tip_delta_tc", "kulfan_tip_delta_camber",
]

# Groupes fonctionnels
GROUPS = {
    "Planform":     VAR_NAMES[0:4],
    "Centre-corps": VAR_NAMES[4:12],
    "Twist":        VAR_NAMES[12:17],
    "Dièdre":       VAR_NAMES[17:22],
    "Kulfan CST":   VAR_NAMES[22:30],
}

# DataFrame des design variables
df_X = pd.DataFrame(X_raw, columns=VAR_NAMES)

# DataFrame des résultats (scalaires + booléens, hors constraints)
SCALAR_KEYS = [k for k in results_list[0] if k not in ("constraints", "duct_fits", "is_feasible", "success")]
BOOL_KEYS = ["duct_fits", "is_feasible", "success"]
CONSTRAINT_KEYS = list(results_list[0]["constraints"].keys())

df_res = pd.DataFrame([{k: r[k] for k in SCALAR_KEYS} for r in results_list])
for bk in BOOL_KEYS:
    df_res[bk] = [r[bk] for r in results_list]
df_con = pd.DataFrame([r["constraints"] for r in results_list])

# DataFrame complet
df = pd.concat([df_X, df_res, df_con], axis=1)
df["is_feasible"] = df["is_feasible"].astype(bool)

print(f"DataFrame complet : {df.shape}")
print(f"Faisables : {df['is_feasible'].sum()} / {len(df)} ({100*df['is_feasible'].mean():.2f}%)")
df.head()

## 2. Statistiques descriptives — Variables de design (X)

In [ ]:
# Statistiques descriptives des 30 variables de design
desc_X = df_X.describe().T
desc_X["skew"] = df_X.skew()
desc_X["kurtosis"] = df_X.kurtosis()
desc_X.style.background_gradient(cmap="coolwarm", subset=["skew", "kurtosis"]).format("{:.4f}")

In [ ]:
# ── Distributions des variables de design par groupe fonctionnel ──
for group_name, vars_in_group in GROUPS.items():
    n = len(vars_in_group)
    ncols = min(4, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
    axes = np.atleast_2d(axes)
    fig.suptitle(f"Distributions — {group_name}", fontsize=14, fontweight="bold", y=1.02)
    for i, var in enumerate(vars_in_group):
        ax = axes.flat[i]
        ax.hist(df_X[var], bins=50, edgecolor="white", linewidth=0.5, alpha=0.85)
        ax.set_title(var, fontsize=10)
        ax.axvline(df_X[var].mean(), color="red", ls="--", lw=1, label="mean")
        ax.axvline(df_X[var].median(), color="orange", ls="-.", lw=1, label="median")
        if i == 0:
            ax.legend(fontsize=7)
    for j in range(i+1, nrows*ncols):
        axes.flat[j].set_visible(False)
    plt.tight_layout()
    plt.show()

## 3. Statistiques descriptives — Réponses (outputs aéro & performances)

In [ ]:
# Statistiques descriptives des outputs
desc_res = df_res[SCALAR_KEYS].describe().T
desc_res["skew"] = df_res[SCALAR_KEYS].skew()
desc_res["kurtosis"] = df_res[SCALAR_KEYS].kurtosis()
desc_res["%_nan"] = df_res[SCALAR_KEYS].isna().mean() * 100
desc_res["%_inf"] = (np.isinf(df_res[SCALAR_KEYS].values)).mean(axis=0) * 100
desc_res.style.background_gradient(cmap="RdYlGn_r", subset=["skew", "kurtosis"]).format("{:.4g}")

In [ ]:
# ── Distributions des outputs clés ──
KEY_OUTPUTS = ["L_over_D", "CD0", "CDi", "CL", "CM", "static_margin",
               "oswald_e", "alpha_eq", "endurance_min", "range_km",
               "struct_mass", "internal_volume", "Vs", "penalty"]

fig, axes = plt.subplots(4, 4, figsize=(18, 14))
fig.suptitle("Distributions des outputs clés (log-scale si besoin)", fontsize=14, fontweight="bold")
for i, var in enumerate(KEY_OUTPUTS):
    ax = axes.flat[i]
    vals = df_res[var].replace([np.inf, -np.inf], np.nan).dropna()
    # log scale si range > 4 ordres de grandeur
    use_log = (vals.max() - vals.min()) > 0 and np.log10(vals.clip(lower=1e-30).abs().max() / vals.clip(lower=1e-30).abs()[vals.abs() > 0].min()) > 4
    if use_log and (vals > 0).all():
        ax.hist(np.log10(vals), bins=60, edgecolor="white", linewidth=0.3, color="steelblue")
        ax.set_xlabel(f"log₁₀({var})")
    else:
        ax.hist(vals, bins=60, edgecolor="white", linewidth=0.3, color="steelblue")
        ax.set_xlabel(var)
    ax.set_ylabel("count")
    ax.set_title(var, fontsize=10)
for j in range(len(KEY_OUTPUTS), 16):
    axes.flat[j].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Analyse de faisabilité & contraintes

Quelles contraintes sont les plus violées ? Quel est le taux de satisfaction par contrainte ?

In [ ]:
# ── Taux de satisfaction par contrainte ──
# Convention : g_i <= 0 → faisable
constraint_satisfied = (df_con <= 0).mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#e74c3c" if v < 0.5 else "#f39c12" if v < 0.8 else "#2ecc71" for v in constraint_satisfied.values]
constraint_satisfied.plot.barh(ax=ax, color=colors, edgecolor="white")
ax.set_xlabel("Taux de satisfaction (g ≤ 0)")
ax.set_title("Taux de satisfaction par contrainte", fontweight="bold")
ax.axvline(1.0, color="green", ls="--", alpha=0.4)
ax.axvline(0.5, color="red", ls="--", alpha=0.4)
for i, (v, name) in enumerate(zip(constraint_satisfied.values, constraint_satisfied.index)):
    ax.text(v + 0.01, i, f"{v:.1%}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

print("\n── Contraintes les plus restrictives ──")
print(constraint_satisfied.head(5).to_string())

In [ ]:
# ── Nombre de contraintes violées par design ──
n_violated = (df_con > 0).sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogramme
axes[0].hist(n_violated, bins=range(0, n_violated.max()+2), edgecolor="white",
             color="steelblue", alpha=0.85, align="left")
axes[0].set_xlabel("Nombre de contraintes violées")
axes[0].set_ylabel("Nombre de designs")
axes[0].set_title("Distribution du nombre de violations", fontweight="bold")
axes[0].axvline(0.5, color="green", ls="--", lw=2, label="Faisable (0)")
axes[0].legend()

# Heatmap : quelles paires de contraintes sont co-violées
violated_binary = (df_con > 0).astype(int)
co_violation = violated_binary.T @ violated_binary / len(violated_binary)
mask = np.triu(np.ones_like(co_violation, dtype=bool), k=1)
sns.heatmap(co_violation, mask=mask, cmap="Reds", annot=True, fmt=".2f",
            ax=axes[1], square=True, cbar_kws={"label": "Taux co-violation"})
axes[1].set_title("Co-violation des contraintes", fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution des valeurs de contrainte (violation = positif) ──
fig, axes = plt.subplots(3, 5, figsize=(20, 10))
fig.suptitle("Distributions des valeurs de contrainte (g > 0 = violée)", fontsize=14, fontweight="bold")
for i, c in enumerate(CONSTRAINT_KEYS):
    ax = axes.flat[i]
    vals = df_con[c].replace([np.inf, -np.inf], np.nan).dropna()
    # Clipper pour lisibilité
    q01, q99 = vals.quantile(0.01), vals.quantile(0.99)
    vals_clip = vals.clip(q01, q99)
    ax.hist(vals_clip, bins=60, edgecolor="white", linewidth=0.3, color="steelblue", alpha=0.8)
    ax.axvline(0, color="red", ls="--", lw=1.5)
    pct_ok = (vals <= 0).mean()
    ax.set_title(f"{c}\n({pct_ok:.0%} OK)", fontsize=9)
for j in range(len(CONSTRAINT_KEYS), 15):
    axes.flat[j].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Boxplots & détection d'outliers

In [ ]:
# ── Boxplots normalisés des variables de design ──
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_norm = pd.DataFrame(scaler.fit_transform(df_X), columns=VAR_NAMES)

fig, ax = plt.subplots(figsize=(18, 6))
X_norm.boxplot(ax=ax, vert=True, patch_artist=True,
               boxprops=dict(facecolor="lightblue", alpha=0.7),
               medianprops=dict(color="red", linewidth=2))
ax.set_xticklabels(VAR_NAMES, rotation=90, fontsize=8)
ax.set_title("Boxplots des variables de design (normalisées [0,1])", fontweight="bold")
ax.set_ylabel("Valeur normalisée")
plt.tight_layout()
plt.show()

In [ ]:
# ── Outliers dans les outputs (IQR method) ──
def count_outliers_iqr(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return ((series < Q1 - 1.5*IQR) | (series > Q3 + 1.5*IQR)).sum()

outlier_counts = df_res[SCALAR_KEYS].apply(count_outliers_iqr).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 5))
outlier_counts.plot.bar(ax=ax, color="coral", edgecolor="white")
ax.set_ylabel("Nombre d'outliers (IQR)")
ax.set_title("Outliers par variable de sortie (méthode IQR)", fontweight="bold")
ax.axhline(len(df)*0.05, color="gray", ls="--", label="5% du dataset")
ax.legend()
plt.xticks(rotation=60, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

## 6. Corrélations

### 6a. Corrélations entre variables de design (inputs)

In [ ]:
# ── Matrice de corrélation des inputs ──
corr_X = df_X.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_X, dtype=bool), k=1)
sns.heatmap(corr_X, mask=mask, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", ax=ax, square=True,
            annot_kws={"size": 6}, linewidths=0.5)
ax.set_title("Corrélations entre variables de design", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

# Paires fortement corrélées
high_corr = []
for i in range(len(VAR_NAMES)):
    for j in range(i+1, len(VAR_NAMES)):
        r = corr_X.iloc[i, j]
        if abs(r) > 0.3:
            high_corr.append((VAR_NAMES[i], VAR_NAMES[j], r))
if high_corr:
    print("Paires avec |r| > 0.3 :")
    for a, b, r in sorted(high_corr, key=lambda x: -abs(x[2])):
        print(f"  {a} ↔ {b} : r = {r:.3f}")
else:
    print("Aucune corrélation forte entre inputs (attendu pour un DOE)")

### 6b. Corrélations inputs → outputs clés (Spearman + Pearson)

In [ ]:
# ── Corrélations de Spearman : inputs → outputs clés ──
from scipy.stats import spearmanr

TARGET_OUTPUTS = ["L_over_D", "CD0", "static_margin", "oswald_e",
                  "endurance_min", "struct_mass", "Vs", "penalty"]

# Calculer les corrélations de Spearman
spearman_matrix = np.zeros((len(VAR_NAMES), len(TARGET_OUTPUTS)))
for j, out in enumerate(TARGET_OUTPUTS):
    y = df_res[out].replace([np.inf, -np.inf], np.nan)
    valid = y.notna()
    for i, var in enumerate(VAR_NAMES):
        if valid.sum() > 10:
            rho, _ = spearmanr(df_X.loc[valid, var], y[valid])
            spearman_matrix[i, j] = rho

df_spearman = pd.DataFrame(spearman_matrix, index=VAR_NAMES, columns=TARGET_OUTPUTS)

fig, ax = plt.subplots(figsize=(12, 14))
sns.heatmap(df_spearman, cmap="RdBu_r", center=0, vmin=-0.6, vmax=0.6,
            annot=True, fmt=".2f", ax=ax, linewidths=0.5,
            annot_kws={"size": 8})
ax.set_title("Corrélations de Spearman : Variables de design → Outputs clés", fontweight="bold")
ax.set_ylabel("Variable de design")
ax.set_xlabel("Output")
plt.tight_layout()
plt.show()

# Top drivers par output
print("\n── Top 3 drivers par output (|ρ| max) ──")
for out in TARGET_OUTPUTS:
    top3 = df_spearman[out].abs().nlargest(3)
    drivers = ", ".join([f"{n} ({df_spearman.loc[n, out]:+.3f})" for n in top3.index])
    print(f"  {out:20s} : {drivers}")

## 7. Analyse en composantes principales (PCA)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Standardiser les inputs
X_scaled = StandardScaler().fit_transform(df_X)
pca = PCA().fit(X_scaled)

# ── Variance expliquée ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(range(1, 31), pca.explained_variance_ratio_, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Composante principale")
axes[0].set_ylabel("Variance expliquée")
axes[0].set_title("Scree plot", fontweight="bold")

# Variance cumulée
cumvar = np.cumsum(pca.explained_variance_ratio_)
axes[1].plot(range(1, 31), cumvar, "o-", color="steelblue", markersize=5)
axes[1].axhline(0.90, color="red", ls="--", label="90%")
axes[1].axhline(0.95, color="orange", ls="--", label="95%")
axes[1].axhline(0.99, color="green", ls="--", label="99%")
n90 = np.searchsorted(cumvar, 0.90) + 1
n95 = np.searchsorted(cumvar, 0.95) + 1
axes[1].set_xlabel("Nombre de composantes")
axes[1].set_ylabel("Variance cumulée")
axes[1].set_title(f"Variance cumulée (90% → {n90} PC, 95% → {n95} PC)", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Projection PCA 2D colorée par faisabilité et penalty ──
X_pca = pca.transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Coloré par faisabilité
feasible = df["is_feasible"].values
axes[0].scatter(X_pca[~feasible, 0], X_pca[~feasible, 1], s=3, alpha=0.15, c="gray", label="Infaisable")
axes[0].scatter(X_pca[feasible, 0], X_pca[feasible, 1], s=25, alpha=0.9, c="lime", edgecolors="darkgreen", linewidth=0.5, label="Faisable")
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
axes[0].set_title("PCA — Faisabilité", fontweight="bold")
axes[0].legend(markerscale=3)

# Coloré par log(penalty)
penalty = df_res["penalty"].replace([np.inf, -np.inf], np.nan).fillna(0).values
log_pen = np.log10(penalty.clip(min=1))
sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], s=3, alpha=0.3, c=log_pen, cmap="hot_r")
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
axes[1].set_title("PCA — log₁₀(penalty)", fontweight="bold")
plt.colorbar(sc, ax=axes[1], label="log₁₀(penalty)")

plt.tight_layout()
plt.show()

In [ ]:
# ── Loadings PC1 & PC2 : quelles variables contribuent le plus ? ──
loadings = pd.DataFrame(pca.components_[:3].T, index=VAR_NAMES, columns=["PC1", "PC2", "PC3"])

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(VAR_NAMES))
width = 0.25
ax.bar(x - width, loadings["PC1"], width, label="PC1", color="steelblue")
ax.bar(x, loadings["PC2"], width, label="PC2", color="coral")
ax.bar(x + width, loadings["PC3"], width, label="PC3", color="seagreen")
ax.set_xticks(x)
ax.set_xticklabels(VAR_NAMES, rotation=90, fontsize=8)
ax.set_ylabel("Loading")
ax.set_title("PCA Loadings — 3 premières composantes", fontweight="bold")
ax.legend()
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## 8. Analyse Faisable vs Infaisable

Comparaison des distributions des variables de design entre designs faisables et infaisables.

In [ ]:
# ── Comparaison faisable vs infaisable : moyennes normalisées ──
df_feas = df_X[df["is_feasible"]]
df_infeas = df_X[~df["is_feasible"]]

# Test de Mann-Whitney U pour chaque variable
from scipy.stats import mannwhitneyu

mw_results = []
for var in VAR_NAMES:
    if len(df_feas) >= 5:
        stat, pval = mannwhitneyu(df_feas[var], df_infeas[var], alternative="two-sided")
        effect_size = (df_feas[var].mean() - df_infeas[var].mean()) / df_X[var].std()
        mw_results.append({"variable": var, "p_value": pval, "effect_size": effect_size,
                           "mean_feasible": df_feas[var].mean(), "mean_infeasible": df_infeas[var].mean()})

df_mw = pd.DataFrame(mw_results).set_index("variable").sort_values("p_value")
df_mw["significant"] = df_mw["p_value"] < 0.05

print(f"Variables significativement différentes (p < 0.05) : {df_mw['significant'].sum()} / {len(VAR_NAMES)}")
df_mw.style.background_gradient(cmap="RdYlGn_r", subset=["p_value"]).format({
    "p_value": "{:.2e}", "effect_size": "{:.3f}", "mean_feasible": "{:.4f}", "mean_infeasible": "{:.4f}"
})

In [ ]:
# ── Violin plots : faisable vs infaisable pour les variables les plus discriminantes ──
top_vars = df_mw[df_mw["significant"]].head(12).index.tolist()
if len(top_vars) == 0:
    top_vars = VAR_NAMES[:12]  # fallback

n = len(top_vars)
ncols = 4
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4*nrows))
fig.suptitle("Distributions : Faisable vs Infaisable (top variables discriminantes)", fontsize=14, fontweight="bold")

for i, var in enumerate(top_vars):
    ax = axes.flat[i]
    data_plot = pd.DataFrame({
        "value": pd.concat([df_feas[var], df_infeas[var]]),
        "Faisable": ["Oui"]*len(df_feas) + ["Non"]*len(df_infeas)
    })
    sns.violinplot(data=data_plot, x="Faisable", y="value", ax=ax, palette={"Oui": "limegreen", "Non": "lightcoral"}, inner="quartile")
    ax.set_title(f"{var}\n(p={df_mw.loc[var, 'p_value']:.1e})", fontsize=9)
    ax.set_xlabel("")

for j in range(len(top_vars), nrows*ncols):
    axes.flat[j].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Performances des designs faisables ──
if df["is_feasible"].sum() > 0:
    df_f = df[df["is_feasible"]].copy()
    perf_cols = ["L_over_D", "endurance_min", "range_km", "Vs", "struct_mass", 
                 "static_margin", "oswald_e", "CD0", "alpha_eq", "penalty"]
    perf_available = [c for c in perf_cols if c in df_f.columns]
    
    print(f"── Statistiques des {len(df_f)} designs faisables ──\n")
    print(df_f[perf_available].describe().T.to_string())
    
    # Meilleur L/D
    best_idx = df_f["L_over_D"].idxmax()
    print(f"\n── Meilleur L/D = {df_f.loc[best_idx, 'L_over_D']:.4f} (index {best_idx}) ──")
    print(f"   endurance = {df_f.loc[best_idx, 'endurance_min']:.4g} min")
    print(f"   range     = {df_f.loc[best_idx, 'range_km']:.4g} km")
    print(f"   Vs        = {df_f.loc[best_idx, 'Vs']:.2f} m/s")
    print(f"   SM        = {df_f.loc[best_idx, 'static_margin']:.3f}")
else:
    print("Aucun design faisable dans le dataset.")

## 9. Visualisations avancées

In [ ]:
# ── Parallel coordinates : designs faisables (variables normalisées) ──
from pandas.plotting import parallel_coordinates

if df["is_feasible"].sum() >= 2:
    df_feas_norm = X_norm[df["is_feasible"]].copy()
    df_feas_norm["L_over_D_bin"] = pd.qcut(
        df_res.loc[df["is_feasible"], "L_over_D"], q=3, labels=["Low", "Mid", "High"]
    )
    
    fig, ax = plt.subplots(figsize=(20, 7))
    parallel_coordinates(df_feas_norm[GROUPS["Planform"] + GROUPS["Centre-corps"] + ["L_over_D_bin"]],
                         "L_over_D_bin", ax=ax, colormap="coolwarm", alpha=0.7, linewidth=1.5)
    ax.set_title("Parallel Coordinates — Designs faisables (Planform + Centre-corps)", fontweight="bold")
    ax.legend(loc="upper right")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Pas assez de designs faisables pour le parallel coordinates.")

In [ ]:
# ── Scatter matrix : variables planform colorées par L/D (faisables) ──
if df["is_feasible"].sum() >= 10:
    df_scatter = df_feas_norm[GROUPS["Planform"]].copy()
    df_scatter["L_over_D"] = df_res.loc[df["is_feasible"], "L_over_D"].values
    
    g = sns.pairplot(df_scatter, hue=None, diag_kind="kde", 
                     plot_kws={"alpha": 0.6, "s": 30, "edgecolor": "white", "linewidth": 0.3},
                     diag_kws={"fill": True})
    g.figure.suptitle("Pairplot — Planform (designs faisables)", y=1.02, fontweight="bold")
    plt.show()
else:
    print("Pas assez de designs faisables pour le pairplot.")

In [ ]:
# ── Feature importance avec Random Forest pour prédire la faisabilité ──
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_scaled, df["is_feasible"].astype(int))

importances = pd.Series(rf.feature_importances_, index=VAR_NAMES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
importances.plot.barh(ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Importance (Gini)")
ax.set_title(f"Feature Importance — Random Forest → Faisabilité\n(accuracy OOB ≈ {rf.oob_score_ if hasattr(rf, 'oob_score_') else 'N/A'})",
             fontweight="bold")
plt.tight_layout()
plt.show()

print("\nTop 5 variables les plus importantes pour la faisabilité :")
for name, imp in importances.tail(5).items():
    print(f"  {name:25s} : {imp:.4f}")

In [ ]:
# ── Scatter 2D des 2 variables les plus importantes (faisable/infaisable) ──
top2 = importances.tail(2).index.tolist()

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(df_X.loc[~df["is_feasible"], top2[1]], df_X.loc[~df["is_feasible"], top2[0]],
           s=5, alpha=0.15, c="gray", label="Infaisable")
ax.scatter(df_X.loc[df["is_feasible"], top2[1]], df_X.loc[df["is_feasible"], top2[0]],
           s=40, alpha=0.9, c="lime", edgecolors="darkgreen", linewidth=0.5, label="Faisable")
ax.set_xlabel(top2[1])
ax.set_ylabel(top2[0])
ax.set_title(f"Espace de design : {top2[1]} vs {top2[0]}", fontweight="bold")
ax.legend(markerscale=2)
plt.tight_layout()
plt.show()

## 10. Corrélations entre outputs & trade-offs

In [ ]:
# ── Corrélations entre outputs (sur les designs faisables) ──
PERF_OUTPUTS = ["L_over_D", "CD0", "CDi", "oswald_e", "static_margin", 
                "endurance_min", "range_km", "Vs", "struct_mass", "alpha_eq", "S_ref", "AR"]
available_perf = [c for c in PERF_OUTPUTS if c in df_res.columns]

if df["is_feasible"].sum() >= 10:
    corr_out = df_res.loc[df["is_feasible"], available_perf].corr(method="spearman")
else:
    corr_out = df_res[available_perf].corr(method="spearman")

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_out, dtype=bool), k=1)
sns.heatmap(corr_out, mask=mask, cmap="RdBu_r", center=0, annot=True, fmt=".2f",
            ax=ax, square=True, linewidths=0.5)
ax.set_title("Corrélations Spearman entre outputs (designs faisables)" if df["is_feasible"].sum() >= 10 
             else "Corrélations Spearman entre outputs (tous)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Trade-off plots pour les designs faisables ──
if df["is_feasible"].sum() >= 5:
    df_f = df[df["is_feasible"]].copy()
    
    trade_offs = [
        ("L_over_D", "struct_mass", "Finesse vs Masse"),
        ("L_over_D", "static_margin", "Finesse vs Marge statique"),
        ("endurance_min", "Vs", "Endurance vs Vitesse de décrochage"),
        ("S_ref", "CD0", "Surface de référence vs CD0"),
    ]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    for ax, (x, y, title) in zip(axes.flat, trade_offs):
        sc = ax.scatter(df_f[x], df_f[y], s=40, c=df_f["L_over_D"], cmap="viridis",
                        edgecolors="black", linewidth=0.3, alpha=0.85)
        ax.set_xlabel(x)
        ax.set_ylabel(y)
        ax.set_title(title, fontweight="bold")
        plt.colorbar(sc, ax=ax, label="L/D")
    plt.suptitle("Trade-offs — Designs faisables", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("Pas assez de designs faisables pour les trade-offs.")

## 11. Résumé & observations clés

In [ ]:
# ── Résumé automatique ──
n_total = len(df)
n_feas = df["is_feasible"].sum()

print("=" * 70)
print("                    RÉSUMÉ DE L'ANALYSE")
print("=" * 70)
print(f"\n  Dataset : {n_total} évaluations, 30 variables de design")
print(f"  Faisabilité : {n_feas} / {n_total} ({100*n_feas/n_total:.2f}%)")
print(f"  Taux de succès (simulation) : {df['success'].mean():.1%}")

print(f"\n── Contraintes les plus restrictives ──")
for name, rate in constraint_satisfied.head(5).items():
    print(f"  {name:25s} : {rate:.1%} satisfaite")

print(f"\n── Variables de design : couverture ──")
print(f"  Distributions quasi-uniformes (DOE type LHS)")
print(f"  PCA : {n90} composantes pour 90% de variance, {n95} pour 95%")

if n_feas > 0:
    best_idx = df.loc[df["is_feasible"], "L_over_D"].idxmax()
    print(f"\n── Meilleur design faisable (index {best_idx}) ──")
    print(f"  L/D        = {df.loc[best_idx, 'L_over_D']:.4f}")
    print(f"  endurance  = {df.loc[best_idx, 'endurance_min']:.4g} min")
    print(f"  range      = {df.loc[best_idx, 'range_km']:.4g} km")

print(f"\n── Top 5 drivers de faisabilité (Random Forest) ──")
for name, imp in importances.tail(5).items():
    print(f"  {name:25s} : importance = {imp:.4f}")

print("\n" + "=" * 70)